# Experimente ausführen
Jeder Aufruf erstellt einen eigenen Run mit frischem Modell und denselben gespeicherten Splits.
Feste Reihenfolge: **Resampling → CDA → CLP**. Jede Methode ist optional.
Bei CDA + CLP werden auch geeignete CDA-Kopien für CLP gepaart.

Paketinstallation, Repository-Clone und ggf. Drive-Mount einmalig vor diesem Notebook erledigen.
Die vorhandenen Analyse-Notebooks lesen anschließend die gespeicherten Ergebnisse.

In [ ]:
from pathlib import Path
import os

# Lokal aus dem Projekt oder notebooks/ starten. Auf Colab/Kaggle ggf. explizit setzen.
repo = Path.cwd()
if not (repo / "src").is_dir():
    repo = repo.parent
if not (repo / "src").is_dir():
    raise RuntimeError("repo auf den Pfad zum gender-bias-hate-speech Repository setzen")
os.chdir(repo)
import sys
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from src.config import ExperimentConfig
from src.experiments import run_experiment
import pandas as pd

results_root = repo / "results"  # Auf Colab ggf. einen gemounteten Drive-Pfad verwenden


## Läufe auswählen
Standardmäßig wird nur die Baseline ausgeführt. Mit `RUN_ALL = True` werden alle acht Kombinationen ausgewählt.
`SEEDS` steuert Wiederholungen; der Split-Seed aus der Dataset-Config bleibt gleich.
Für Hyperparameterwahl `RUN_TEST = False` verwenden; die Validation wird weiterhin ausgewertet.
`truncation=True` und `max_length=512` gelten einheitlich für alle Varianten.
Die YAMLs der einzelnen Methoden lassen sich konfliktfrei kombinieren, weil sie verschiedene Blöcke setzen.

In [ ]:
from itertools import combinations

RUN_ALL = False
RUN_TEST = True
SEEDS = [42]

methods = ("resampling", "cda", "clp")
combinations_to_run = (
    [combo for n in range(4) for combo in combinations(methods, n)]
    if RUN_ALL else [()]  # z.B. [("cda", "clp")] für einen kombinierten Lauf
)

experiments = []
for seed in SEEDS:
    for combo in combinations_to_run:
        variant = "_".join(combo) or "baseline"
        config = ExperimentConfig.from_files(
            model="configs/model_full_finetune.yaml",
            training="configs/training.yaml",
            evaluation="configs/evaluation.yaml",
            dataset="configs/datasets/3D_combined.yaml",
            mitigations=[f"configs/mitigation/{method}.yaml" for method in combo],
        )
        # Änderungen VOR run_experiment werden im Config-Snapshot gespeichert.
        values = config.to_dict()
        values["training"]["seed"] = seed
        values["tokenization"] = {"max_length": 512, "truncation": True}
        values["evaluation"]["run_test"] = RUN_TEST
        # Optional auf geeigneter GPU: values["training"]["fp16"] = True
        config = ExperimentConfig(values, list(config.sources))
        experiments.append((f"3D_combined_{variant}_seed{seed}", config))

pd.DataFrame([{"experiment": name, "mitigation": cfg.get("mitigation", {})}
              for name, cfg in experiments])


## Training starten
Diese Zelle startet alle ausgewählten Läufe nacheinander. Erneutes Ausführen erzeugt neue Run-Verzeichnisse.
Bei einem Fehler stoppt die Schleife; bereits abgeschlossene Runs bleiben gespeichert.
Die Ursache steht auch in `status.json` des fehlgeschlagenen Runs.

In [ ]:
results = []
for name, config in experiments:
    result = run_experiment(config, name, results_root=results_root)
    results.append(result)
    print("Abgeschlossen:", result["run_dir"])


In [ ]:
summary = pd.DataFrame([
    {"experiment": result["experiment_name"], "run_dir": result["run_dir"],
     **result["validation_metrics"]}
    for result in results
])
display(summary)


## Gespeicherte Ergebnisse
Jeder Run enthält `config.yaml`, Split-Snapshots, `mitigation_report.json`, `training_rows.csv`,
Modell/Tokenizer, Trainingshistorie und Validation-Metriken. Mit `RUN_TEST=True` entstehen zusätzlich
Original-/Swap-Predictions, gepaarte Predictions und die Gruppen-Evaluation.
Bootstrap und ausführliche Counterfactual-Auswertung bleiben im Analyse-Notebook.

`training_rows.csv`: `id` identifiziert die Trainingszeile, `source_id` ihren direkten Vorgänger,
`root_id` die Zeile vor allen Mitigations. CLP ergänzt keine Zeilen.
Bei CLP enthält der Trainingsloss CE + Pairing, der Validation-Loss nur CE.
Die implementierte CLP-Variante unterstützt zwei Output-Logits. Equalized-Odds-Training ist noch nicht implementiert.